## Combining Nullable and Optional

Wie hängen „darf fehlen“ und „darf `None` sein“ zusammen? Die beiden Eigenschaften sind unabhängig voneinander: Ob ein Field fehlen darf, entscheidet der Default. Ob `None` ein gültiger Wert ist, entscheidet der Typ. Daraus ergeben sich vier Kombinationen, die dieses Notebook nacheinander durchgeht.

**Danach ist klar:**
- welche vier Kombinationen aus Pflicht/optional und nullable/nicht nullable es gibt,
- wie jede Kombination annotiert wird,
- dass ein Default nur greift, wenn das Field fehlt, nicht wenn `None` übergeben wird.

Voraussetzung: Notebook `07 - Nullable Fields`.

### Required, Not Nullable

Das Field muss angegeben werden, und `None` ist kein gültiger Wert.

In [ ]:
# `BaseModel` definiert das Model, `ValidationError` fängt später die Fehlerfälle ab.
from pydantic import BaseModel, ValidationError

In [ ]:
# Kein Default und kein `None` im Typ: Das Field ist Pflicht und nicht nullable.
class Model(BaseModel):
    field: int

In [ ]:
# Ein übergebenes `None` wird abgelehnt (`int_type`), weil der Typ nur `int` erlaubt.
try:
    Model(field=None)
except ValidationError as exc:
    print(exc)

### Required, Nullable

Das Field muss angegeben werden, darf aber `None` sein.

In [ ]:
# `| None` erlaubt `None`, ein Default fehlt, das Field bleibt also Pflicht.
class Model(BaseModel):
    field: int | None

**Vorhersage:** Das Field ist nullable, hat aber keinen Default. Was passiert bei `Model()`?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Pydantic meldet `missing`. Nullable heißt nur, dass `None` ein gültiger Wert ist. Das Field fehlen zu lassen erlaubt erst ein Default.

Kurz gesagt: `None` angeben ist erlaubt, nichts angeben nicht.
</details>

In [ ]:
try:
    Model()
except ValidationError as exc:
    print(exc)

In [ ]:
# Ausdrücklich übergeben ist `None` dagegen gültig.
Model(field=None)

### Optional, Not Nullable

Das Field darf fehlen, `None` ist aber kein gültiger Wert.

In [ ]:
# Der Default `0` macht das Field optional, der Typ `int` erlaubt weiterhin kein `None`.
class Model(BaseModel):
    field: int = 0

In [ ]:
# Ohne Angabe greift der Default `0`.
Model()

**Vorhersage:** Der Default ist `0`. Ersetzt Pydantic ein übergebenes `None` durch den Default, oder wird es abgelehnt?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Pydantic lehnt `None` ab (`int_type`). Der Default greift nur, wenn das Field fehlt, nicht wenn ein Wert ausdrücklich übergeben wird.

Kurz gesagt: Ein Default ersetzt nur eine fehlende Angabe, nie einen ungültigen Wert.
</details>

In [ ]:
try:
    Model(field=None)
except ValidationError as exc:
    print(exc)

### Optional, Nullable

Das Field darf fehlen und darf `None` sein. Dafür sind Typ und Default nötig.

In [ ]:
# `| None` erlaubt `None`, der Default `None` erlaubt das Weglassen.
class Model(BaseModel):
    field: int | None = None

In [ ]:
# Ohne Angabe greift der Default `None`.
Model()

In [ ]:
# Ein ausdrücklich übergebenes `None` ist ebenfalls gültig.
Model(field=None)

### Zusammenfassung

- Pflicht und nicht nullable: `field: int`.
- Pflicht und nullable: `field: int | None`, `None` muss ausdrücklich übergeben werden.
- Optional und nicht nullable: `field: int = 0`, das Field darf fehlen, `None` wird abgelehnt.
- Optional und nullable: `field: int | None = None`.
- Ein Default greift nur bei fehlender Angabe, ein übergebenes `None` ersetzt er nicht.

Weiter mit `09 - Inspecting Fields`.